# Friend A — B32 division-pair model

Единый возобновляемый ноутбук для последней модельной линии: широкая воронка кандидатов и классификатор полной тройки **родитель + две дочери**. Детектор, обычный линкер и число узлов не меняются.

**Текущий результат:** полный Step 1 уже измерен и честно остановлен: зарегистрированный пул покрывает 58/151 = 38.4% GT-делений, а потолок по наличию родителя и обеих дочерей — 121/151 = 80.1%, ниже обязательных 90%. Поэтому обучение не запускается. Подробности: `experiments/local_friend_a_b32_divpair/results/b32_step1_candidate_audit.md`.

Порядок автоматический: candidate audit → фиксированный crop-cache → две честные embryo-fold → полный OOF scoring → официальный pass-2 grid → зарегистрированные ворота → финальная модель только при победе.

## Остановка

Можно нажать **Interrupt Kernel**. Для мягкой остановки создай `C:\Projects\CellTracker\outputs\local_friend_a_b32_divpair\STOP_REQUESTED`. Сохраняются отдельные видео, partial OOF и каждая эпоха. Удали маркер и снова выполни Run All. На leaderboard ноутбук ничего не отправляет.


In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
candidates = [start, start / '.github-push-worktree']
for parent in start.parents:
    candidates.extend([parent, parent / '.github-push-worktree'])
REPO_ROOT = next((p for p in candidates if (p / 'TEAM.md').is_file() and (p / 'PROMPT_FRIEND_A.md').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Repository root not found. Start Jupyter from C:\\Projects\\CellTracker.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from experiments.local_friend_a_b32_divpair.runner import (
    B32Config, clear_stop, compact_status, preflight, request_stop, run_pipeline,
)
print('Repository:', REPO_ROOT)


In [ ]:
# Настройки можно переопределить через BIOHUB_B32_* до запуска Jupyter.
cfg = B32Config.from_environment(REPO_ROOT)
cfg.serialisable()


In [ ]:
# Удаляется только старый запрос остановки; результаты и checkpoints не трогаются.
clear_stop(cfg)
check = preflight(cfg, gpu_smoke=cfg.run_pipeline)
check


Следующая ячейка — весь долгий возобновляемый прогон. После Step 1 она сама остановится с ошибкой, если зарегистрированный пул покрывает меньше 90% GT-делений; рецептура автоматически не расширяется.


In [ ]:
state = run_pipeline(cfg)
compact_status(cfg)


In [ ]:
status = compact_status(cfg)
result_path = cfg.output_dir / 'result.md'
if result_path.is_file():
    print(result_path.read_text(encoding='utf-8'))
print('Stop marker:', cfg.stop_file)


In [ ]:
# Выполни эту ячейку отдельно только когда нужна мягкая остановка.
REQUEST_STOP = False
if REQUEST_STOP:
    print('Stop requested:', request_stop(cfg))
